In [10]:
# Model loading with file existence checks for robustness
import os
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image

NUM_CLASSES = 12
CLASSES = [
    "Asthma", "Heart Failure", "COPD", "Pneumonia", "Pleural Effusion", "Lung Fibrosis", "Bronchitis",
    "Bronchiectasis", "Bronchiolitis", "URTI", "Normal", "None",
]
THRESHOLD = 0.5
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class MetadataWeightNet(nn.Module):
    def __init__(self, meta_dim=2, num_classes=NUM_CLASSES):
        super().__init__()
        self.importance = nn.Sequential(
            nn.Linear(meta_dim, 16),
            nn.ReLU(),
            nn.Linear(16, meta_dim),
            nn.Softmax(dim=-1),
        )
        self.projection = nn.Linear(meta_dim, num_classes)

    def forward(self, metadata):
        weights           = self.importance(metadata)
        self.last_weights = weights
        weighted          = metadata * weights
        return self.projection(weighted)

# Path constants
CNN_WEIGHTS_PATH  = "Trained_Weights.pth"
META_WEIGHTS_PATH = "metadata_weights.pth"
IMAGE_PATH        = "spectrograms\\130_123_85_0_F_COPD_timestretch.png"  # Use forward slashes for compatibility LLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLLl

# File existence checks
for path in [CNN_WEIGHTS_PATH, META_WEIGHTS_PATH, IMAGE_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file not found: {path}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Construct models
model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3, inplace=True),
    nn.Linear(1280, NUM_CLASSES)
)
meta_model = MetadataWeightNet()

model.load_state_dict(torch.load(CNN_WEIGHTS_PATH,  map_location=device))
meta_model.load_state_dict(torch.load(META_WEIGHTS_PATH, map_location=device))
model.eval()
meta_model.eval()
model      = model.to(device)
meta_model = meta_model.to(device)

print(f"Models loaded — running on {device}")

def parse_age_gender(filename):
    """Read age and gender from filename. Falls back to (0.5, 0.5) if unparseable."""
    try:
        name     = os.path.splitext(os.path.basename(filename))[0]
        parts    = name.split('_', 6)
        age_norm = float(f"{parts[2]}.{parts[3]}") / 100.0
        gender   = 1.0 if parts[4].upper() == 'M' else 0.0
        return age_norm, gender
    except (ValueError, IndexError):
        return 0.5, 0.5

def predict(image_path, age=None, gender=None):
    """
    Run inference on a single PNG.
    Args:
        image_path : path to the spectrogram PNG
        age        : int, e.g. 54 — if None, read from filename
        gender     : 'M' or 'F'  — if None, read from filename
    """
    fname = os.path.basename(image_path)
    # Age / gender — use provided values or fall back to filename parsing
    if age is not None and gender is not None:
        age_norm   = age / 100.0
        gender_val = 1.0 if str(gender).upper() == 'M' else 0.0
    else:
        age_norm, gender_val = parse_age_gender(fname)
    # Build tensors
    img      = Image.open(image_path).convert("RGB")
    tensor   = val_transforms(img).unsqueeze(0).to(device)
    metadata = torch.tensor([[age_norm, gender_val]], dtype=torch.float32).to(device)
    # Forward pass
    with torch.no_grad():
        logits = model(tensor) + meta_model(metadata)
        probs  = torch.sigmoid(logits).squeeze(0)
    # Sort by confidence descending
    return sorted(
        [(CLASSES[i], round(probs[i].item() * 100, 1)) for i in range(NUM_CLASSES)],
        key=lambda x: -x[1]
    )

def print_results(scores, image_path):
    fname      = os.path.basename(image_path)
    predicted  = [cls for cls, pct in scores if pct >= THRESHOLD * 100]
    print(f"\n{'─' * 65}")
    print(f"  File     : {fname}")
    print(f"{'─' * 65}")
    print(f"  {'Disease':<22} {'Confidence':>10}   Bar")
    print(f"  {'─' * 60}")
    for cls, pct in scores:
        bar  = "█" * int(pct / 2.5)
        flag = "  ← POSITIVE" if pct >= THRESHOLD * 100 else ""
        print(f"  {cls:<22}: {pct:>6.1f}%   {bar}{flag}")
    print(f"\n  Predicted : {predicted if predicted else ['Nothing above threshold']}")
    print(f"  (scores are per-class — do not sum to 100%)")
    print(f"{'─' * 65}\n")

# Run — just change the path (and optionally age/gender) here
AGE        = None    # set to an int e.g. 54, or None to read from filename
GENDER     = None    # set to 'M' or 'F',  or None to read from filename
scores = predict(IMAGE_PATH, age=AGE, gender=GENDER)
print_results(scores, IMAGE_PATH)


Models loaded — running on cpu

─────────────────────────────────────────────────────────────────
  File     : 130_123_85_0_F_COPD_timestretch.png
─────────────────────────────────────────────────────────────────
  Disease                Confidence   Bar
  ────────────────────────────────────────────────────────────
  Bronchitis            :   57.0%   ██████████████████████  ← POSITIVE
  Normal                :   48.3%   ███████████████████
  COPD                  :   18.9%   ███████
  Heart Failure         :   12.9%   █████
  Asthma                :   10.6%   ████
  Pneumonia             :    9.8%   ███
  Lung Fibrosis         :    1.5%   
  Pleural Effusion      :    0.6%   
  Bronchiectasis        :    0.3%   
  Bronchiolitis         :    0.2%   
  URTI                  :    0.2%   
  None                  :    0.2%   

  Predicted : ['Bronchitis']
  (scores are per-class — do not sum to 100%)
─────────────────────────────────────────────────────────────────



In [0]:
# Install core dependencies for spectrogram pipeline, as specified in README
!pip install librosa numpy opencv-python matplotlib pandas